# Kangaroo (JeanLucPons) build + smoke test (Colab T4)

Verifies the GPU solver builds and recovers a known key on a small
synthetic puzzle. Run this once before pointing at the real target.

**Why JLP, not Kangaroo-256?** K-256 (the 256-bit fork) is the eventual
production target since puzzle #135 needs a 134-bit interval (JLP caps at
125-bit). But on Colab T4, K-256 misbehaves on narrow intervals (60-70 bit
puzzles never converge, GPU keeps producing ops without finding the key).
We use plain JLP for the smoke test to validate the rest of the pipeline
while we investigate the K-256 issue separately.

**Prereq:** project (`bitcoin-prize`) reachable at `PROJECT_DIR` below.
Easiest paths: `git clone`, upload via Files, or Drive mount.

**Runtime:** Runtime → Change runtime type → T4 GPU.


In [ ]:
# Sanity: confirm a T4 is attached.
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader


In [ ]:
import os, subprocess, sys

PROJECT_DIR = "/content/bitcoin-prize"   # change to your Drive path if needed
KANGAROO_DIR = "/content/Kangaroo-256"
WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.isdir(PROJECT_DIR), (
    f"project not found at {PROJECT_DIR!r}. Either git-clone, upload via "
    f"Files, or set PROJECT_DIR to a Drive path."
)
sys.path.insert(0, PROJECT_DIR)
print("project:", PROJECT_DIR)


In [ ]:
# Clone + build JeanLucPons/Kangaroo. ccap=75 = T4 (compute capability 7.5).
# JLP's Makefile hardcodes ancient CUDA 8.0 / g++-4.8 paths; we override.
KANGAROO_DIR = "/content/Kangaroo"

if not os.path.isdir(KANGAROO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/JeanLucPons/Kangaroo.git", KANGAROO_DIR],
        check=True,
    )

subprocess.run(["make", "clean"], cwd=KANGAROO_DIR, check=False,
               capture_output=True)
build = subprocess.run(
    ["make", "gpu=1", "ccap=75",
     "CUDA=/usr/local/cuda", "CXXCUDA=/usr/bin/g++",
     "all"],
    cwd=KANGAROO_DIR, capture_output=True, text=True,
)
print(build.stdout[-2000:])
if build.returncode != 0:
    print("STDERR:", build.stderr[-2000:])
    raise RuntimeError(f"build failed (rc={build.returncode})")

binary = os.path.join(KANGAROO_DIR, "kangaroo")
assert os.path.isfile(binary), f"binary missing after build: {binary}"
print("built:", binary)


In [ ]:
# Generate a small synthetic puzzle with a known answer.
# 60 bits → ~2 G ops → ~4 s on T4 at ~600 MK/s.
# Solve time scales as sqrt(N): each +2 bits ≈ 2× wall time.
PUZZLE_PATH = f"{WORK_DIR}/puzzle.txt"
BITS = 60
SEED = 42

gen = subprocess.run(
    ["python3", "scripts/make_synthetic_puzzle.py",
     "--bits", str(BITS), "--seed", str(SEED), "--out", PUZZLE_PATH],
    cwd=PROJECT_DIR, capture_output=True, text=True, check=True,
)
print(gen.stderr.strip())
print("---")
print(open(PUZZLE_PATH).read())


In [ ]:
# Run JLP-Kangaroo against the synthetic puzzle.
# JLP's auto dp_bits picker is reliable; skip -d.
# -o writes the recovered key cleanly; stdout is dominated by status lines.
import time, json

RESULT_PATH = "/tmp/kangaroo_result.txt"
if os.path.exists(RESULT_PATH):
    os.remove(RESULT_PATH)

with open(PUZZLE_PATH + ".json") as f:
    manifest = json.load(f)
print(f"BITS={BITS}, expecting d = 0x{manifest['d_hex']}")

t0 = time.monotonic()
solve = subprocess.run(
    [binary, "-t", "4", "-gpu", "-gpuId", "0",
     "-o", RESULT_PATH,
     PUZZLE_PATH],
    capture_output=True, text=True, timeout=120,
)
elapsed = time.monotonic() - t0
print(solve.stdout[-2000:])
print(f"--- elapsed {elapsed:.2f}s, rc={solve.returncode}")


In [ ]:
# Verify: parse Priv: line and compare to manifest.
# Prefer the result file (clean) over stdout (cluttered with status lines).
import re

if os.path.exists(RESULT_PATH):
    out = open(RESULT_PATH).read()
    print(f"--- {RESULT_PATH} ---")
    print(out)
else:
    print(f"{RESULT_PATH} missing; falling back to stdout")
    out = solve.stdout

m = re.search(r"Priv\s*:\s*(?:0x)?([0-9a-fA-F]+)", out)
assert m, f"no Priv: line\n--- last 500 chars ---\n{out[-500:]}"
recovered = m.group(1).lower().lstrip("0") or "0"
expected = manifest["d_hex"].lstrip("0") or "0"
assert recovered == expected, f"key mismatch: got {recovered}, expected {expected}"
print(f"PASS — recovered d = 0x{recovered}")


## What this proves

- The Linux/CUDA build path on Colab T4 works (`ccap=75`).
- Kangaroo-256 produces correct output on the JLP-format input we generate.
- The toolchain — clone → build → puzzle gen → solve → verify — is reproducible.

**Next:** build the production-target notebook (`solver.ipynb`) on top of this:
add Drive mount, work-file checkpointing every 20 min, idle-timeout-clean exit.
